# Session 47 — Research Session Log

Creates `research_session_log` table and adds `outcome_label` column to `ref_signal_weights`.

**Run order:**
1. Cell 1 — Create `research_session_log`
2. Cell 2 — Add `outcome_label` column to `ref_signal_weights`
3. Cell 3 — Populate `outcome_label` for all 44 signals
4. Cell 4 — Verify

Cells 1 and 2 are idempotent. Cell 3 uses MERGE so is safe to re-run.

## Cell 1 — Create research_session_log

Delta table in the genealogy schema. One row per person × signal × session.
- `log_id` — UUID generated client-side (avoids a round-trip to get a server-generated ID)
- `week_commencing` — FK to ref_week_plan, passed from the client
- `estimated_score_delta` — base_score × proximity_multiplier × depth_multiplier, calculated at log time
- `confirmed` / `score_delta_actual` — populated in a future session once snapshot comparison is built

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS genealogy.research_session_log (
    log_id                STRING  NOT NULL,
    person_gedcom_id      STRING  NOT NULL,
    signal_code           STRING  NOT NULL,
    session_date          DATE    NOT NULL,
    week_commencing       DATE    NOT NULL,
    branch                STRING  NOT NULL,
    outcome               STRING  NOT NULL,
    notes                 STRING,
    estimated_score_delta DOUBLE,
    confirmed             BOOLEAN,
    score_delta_actual    DOUBLE,
    logged_at             TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Research session log — one row per person x signal x session. Outcome framed as resolved/partial/not_found/deferred. Confirmed and score_delta_actual populated after pipeline run comparison (future).'
""")
print("research_session_log created (or already exists)")

## Cell 2 — Add outcome_label to ref_signal_weights

Imperative outcome-framed label shown in the This Week action list.
e.g. 'Confirm birth' not 'Search birth records'.

In [0]:
# Check if column already exists before adding — idempotent
cols = [c.name for c in spark.table("genealogy.ref_signal_weights").schema.fields]
if "outcome_label" not in cols:
    spark.sql("ALTER TABLE genealogy.ref_signal_weights ADD COLUMN outcome_label STRING")
    print("outcome_label column added")
else:
    print("outcome_label column already exists — skipping ALTER")

## Cell 3 — Populate outcome_label for all 44 signals

Uses MERGE so safe to re-run. Narrative signals included for completeness
even though they are excluded from the action filter in the app.

In [0]:
# outcome_label values — imperative, outcome-framed, max ~30 chars
outcome_labels = [
    # ── Completeness ──────────────────────────────────────────────────────────
    ("SIGNAL_NO_BIRTH_RECORDED",          "Confirm birth"),
    ("SIGNAL_NO_DEATH_RECORDED",          "Confirm death"),
    ("SIGNAL_NO_MARRIAGES",               "Find marriage record"),
    ("SIGNAL_NO_CHILDREN",                "Find children"),
    ("SIGNAL_MISSING_PARENT",             "Find parents"),
    ("SIGNAL_MISSING_CENSUS_COVERAGE",    "Find missing censuses"),
    ("SIGNAL_UNCOVERED_SOURCES",          "Find source documents"),
    ("SIGNAL_DOCS_NOT_TRANSCRIBED",       "Transcribe documents"),
    ("SIGNAL_LATE_LIFE_GAP",              "Fill late life gap"),
    ("SIGNAL_EARLY_LIFE_ONLY",            "Find adult records"),
    ("SIGNAL_CHILD_GAPS",                 "Find gaps between children"),
    ("SIGNAL_UNCONFIRMED_MILITARY",       "Confirm military service"),
    ("SIGNAL_MISSING_OCCUPATION",         "Find occupation records"),
    ("SIGNAL_MISSING_BURIAL",             "Find burial record"),
    ("SIGNAL_NO_RESIDENCE",               "Find residence records"),
    ("SIGNAL_DNA_CITATION_MISSING",       "Add DNA citation"),
    ("SIGNAL_DNA_PATH_INCOMPLETE",        "Complete DNA path"),
    ("SIGNAL_NO_DNA_CORROBORATION",       "Find DNA corroboration"),
    # ── Evidence ──────────────────────────────────────────────────────────────
    ("SIGNAL_NO_DOCUMENTS_AT_ALL",        "Find any documents"),
    ("SIGNAL_FACT_CONFLICT",              "Resolve fact conflicts"),
    ("SIGNAL_TRANSCRIPT_ONLY_FACTS",      "Verify transcript facts"),
    ("SIGNAL_VERY_LOW_EVIDENCE_DENSITY",  "Add more sources"),
    ("SIGNAL_LOW_EVIDENCE_DENSITY",       "Add more sources"),
    ("SIGNAL_SINGLE_SOURCE_DEPENDENCE",   "Find additional sources"),
    ("SIGNAL_UNSOURCED_FAMILY_EVENTS",    "Source family events"),
    ("SIGNAL_IMPRECISE_DATES",            "Confirm precise dates"),
    ("SIGNAL_INCOMPLETE_NAME",            "Complete name record"),
    ("SIGNAL_IMPRECISE_PLACES",           "Confirm precise places"),
    ("SIGNAL_COMMON_ANCESTOR_UNVERIFIED", "Verify common ancestor"),
    # ── Narrative (excluded from action filter but label populated) ───────────
    ("SIGNAL_MIGRANT",                   "Investigate migration"),
    ("SIGNAL_CONFIRMED_MILITARY",         "Research military records"),
    ("SIGNAL_MULTIPLE_SPOUSES",           "Research multiple marriages"),
    ("SIGNAL_YOUNG_DEATH",               "Research early death"),
    ("SIGNAL_LARGE_FAMILY",              "Research family details"),
    ("SIGNAL_NEWSPAPER_MENTION",          "Find newspaper records"),
    ("SIGNAL_WILL_OR_PROBATE",           "Research will or probate"),
    ("SIGNAL_STORY_WRITTEN",             "Story published"),
    ("SIGNAL_TRANSCRIPT_RICH",           "Review transcripts"),
    ("SIGNAL_VARIED_OCCUPATIONS",        "Research occupations"),
    ("SIGNAL_POSSIBLE_RESIDENCE",        "Investigate possible residence"),
    ("SIGNAL_POSSIBLE_MARRIAGE",         "Investigate possible marriage"),
    ("SIGNAL_POSSIBLE_CHILDREN",         "Investigate possible children"),
    # ── Structural (context only — won't appear in action lists) ─────────────
    ("SIGNAL_DIRECT_ANCESTOR",           "Direct ancestor"),
    ("SIGNAL_CLOSE_COLLATERAL",          "Close collateral"),
]

# Build VALUES clause
values_sql = ",\n    ".join(
    f"('{code}', '{label}')"
    for code, label in outcome_labels
)

merge_sql = f"""
MERGE INTO genealogy.ref_signal_weights AS target
USING (
    SELECT col1 AS signal_code, col2 AS outcome_label
    FROM VALUES
    {values_sql}
) AS source
ON target.signal_code = source.signal_code
WHEN MATCHED THEN
    UPDATE SET target.outcome_label = source.outcome_label
"""

spark.sql(merge_sql)
print(f"outcome_label populated for {len(outcome_labels)} signals")

## Cell 4 — Verify

In [0]:
# Verify research_session_log exists and is empty
count = spark.sql("SELECT COUNT(*) AS n FROM genealogy.research_session_log").collect()[0]["n"]
print(f"research_session_log rows: {count}")

# Verify outcome_label populated — show any nulls (should be 0)
nulls = spark.sql("""
    SELECT signal_code
    FROM genealogy.ref_signal_weights
    WHERE outcome_label IS NULL
""").collect()
if nulls:
    print(f"WARNING: {len(nulls)} signals missing outcome_label:")
    for r in nulls:
        print(f"  {r['signal_code']}")
else:
    print("All signals have outcome_label — OK")

# Show sample
display(spark.sql("""
    SELECT signal_code, dimension, outcome_label, base_score
    FROM genealogy.ref_signal_weights
    WHERE dimension IN ('completeness', 'evidence')
    ORDER BY dimension, base_score DESC
"""))